# LeetCode 3033 — Modify the Matrix bằng Python

Link đề: https://leetcode.com/problems/modify-the-matrix/ · Độ khó: **Easy** · Chủ đề: Array, Matrix

Skeleton LeetCode:

```python
from typing import List

class Solution:
    def modifiedMatrix(self, matrix: List[List[int]]) -> List[List[int]]:
        ...
```

Ghi chú: đề và các ví dụ được viết theo trí nhớ về đề gốc; mọi đáp án trong notebook đã được kiểm chứng bằng code, nhưng nên đối chiếu lại với đề gốc nếu muốn khớp từng chữ.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: tạo một ma trận mới `answer` **sao chép** từ `matrix`, rồi **thay mỗi ô `-1`** bằng **giá trị lớn nhất của cột** chứa ô đó.
2. Biết truy cập ma trận hai chiều theo **hàng** (`matrix[i]`) và theo **cột** (cần duyệt `matrix[i][j]` qua mọi `i`, hoặc **chuyển vị** bằng `zip(*matrix)`).
3. Biết **sao chép đúng** một ma trận (từng hàng: `[row[:] for row in matrix]`) và hiểu vì sao sao chép nông (`matrix[:]`) **vẫn dùng chung các hàng**.
4. Đi từ cách **với mỗi ô `-1` quét cả cột** → **tính trước `max` của từng cột một lần** → **`zip(*matrix)` để lấy cột** → **sửa tại chỗ không tạo ma trận mới**, và so sánh đánh đổi giữa chúng.
5. Tránh các lỗi kinh điển: lấy **max theo hàng** thay vì theo cột, **sao chép nông** làm hỏng ma trận gốc, **nhầm `m` và `n`** khi ma trận không vuông, và gọi `zip(matrix)` thay vì `zip(*matrix)`.

## 2. Đề bài & ví dụ

Cho ma trận số nguyên `matrix` kích thước `m x n` (chỉ số từ `0`). Hãy tạo một ma trận `answer` **bằng bản sao** của `matrix`, rồi **thay mọi phần tử `-1` trong `answer`** bằng **giá trị lớn nhất của cột chứa phần tử đó** (lấy từ ma trận gốc). Trả về ma trận `answer`.

Đề đảm bảo **mỗi cột có ít nhất một phần tử không âm** (nên giá trị lớn nhất của cột luôn là một số không âm).

**Ví dụ 1**

- Input: `matrix = [[1, 2, -1], [4, -1, 6], [7, 8, 9]]`
- Output: `[[1, 2, 9], [4, 8, 6], [7, 8, 9]]`
- Giải thích: cột `0` có max `7`, cột `1` có max `8`, cột `2` có max `9`. Ô `(0, 2)` là `-1` được thay bằng max cột `2` (`9`); ô `(1, 1)` là `-1` được thay bằng max cột `1` (`8`).

**Ví dụ 2**

- Input: `matrix = [[3, -1], [5, 2]]`
- Output: `[[3, 2], [5, 2]]`
- Giải thích: ô `(0, 1)` là `-1` thuộc cột `1`, max của cột `1` là `max(-1, 2) = 2`.

**Ràng buộc:** `m == matrix.length`; `n == matrix[i].length`; `2 <= m, n <= 50`; `-1 <= matrix[i][j] <= 100`; mỗi cột của `matrix` có ít nhất một phần tử **không âm**.

In [1]:
from typing import List
import random

examples = [
    ([[1, 2, -1], [4, -1, 6], [7, 8, 9]], [[1, 2, 9], [4, 8, 6], [7, 8, 9]]),
    ([[3, -1], [5, 2]], [[3, 2], [5, 2]]),
    ([[-1, 0, 5], [-1, 1, -1], [4, -1, 2]], [[4, 0, 5], [4, 1, 5], [4, 1, 2]]),
]
for matrix, expected in examples:
    print("matrix =", matrix, "-> kỳ vọng", expected)

matrix = [[1, 2, -1], [4, -1, 6], [7, 8, 9]] -> kỳ vọng [[1, 2, 9], [4, 8, 6], [7, 8, 9]]
matrix = [[3, -1], [5, 2]] -> kỳ vọng [[3, 2], [5, 2]]
matrix = [[-1, 0, 5], [-1, 1, -1], [4, -1, 2]] -> kỳ vọng [[4, 0, 5], [4, 1, 5], [4, 1, 2]]


## 3. Kiến thức nền

### 3.1. Ma trận hai chiều: hàng và cột

`matrix[i]` là **hàng thứ `i`**; `matrix[i][j]` là ô ở **hàng `i`, cột `j`**. Số hàng `m = len(matrix)`, số cột `n = len(matrix[0])`. Muốn lấy **cột `j`** ta phải gom `matrix[0][j], matrix[1][j], ...` qua mọi hàng.

In [2]:
matrix = [[1, 2, -1], [4, -1, 6], [7, 8, 9]]
m, n = len(matrix), len(matrix[0])
print(m, n)                                  # 3 3
print(matrix[1], matrix[1][2])               # [4, -1, 6] 6
col2 = [matrix[i][2] for i in range(m)]
print("cột 2:", col2)                         # [-1, 6, 9]

3 3
[4, -1, 6] 6
cột 2: [-1, 6, 9]


### 3.2. `m` và `n` không cần bằng nhau

Ma trận có thể **không vuông** (`m != n`). Duyệt hàng dùng `range(m)`, duyệt cột dùng `range(n)`. Nhầm hai cận này gây `IndexError` hoặc bỏ sót ô khi ma trận không vuông.

In [3]:
matrix = [[1, 2, 3, 4], [5, 6, 7, 8]]         # 2 hàng, 4 cột
m, n = len(matrix), len(matrix[0])
print("m =", m, "n =", n)
for i in range(m):
    for j in range(n):
        pass
print("duyệt đủ", m * n, "ô")

m = 2 n = 4
duyệt đủ 8 ô


### 3.3. Chuyển vị bằng `zip(*matrix)`

`*matrix` **giải nén** các hàng thành các đối số riêng, rồi `zip` ghép **phần tử thứ `j` của mọi hàng** thành một bộ: đó chính là **cột `j`**. Vì vậy `zip(*matrix)` cho các cột (dưới dạng tuple). Chú ý: `zip(matrix)` (không có `*`) chỉ gói **từng hàng** vào bộ một phần tử.

In [4]:
matrix = [[1, 2, -1], [4, -1, 6], [7, 8, 9]]
print(list(zip(*matrix)))                    # [(1, 4, 7), (2, -1, 8), (-1, 6, 9)]  <- các cột
print(list(zip(matrix)))                     # [([1, 2, -1],), ([4, -1, 6],), ([7, 8, 9],)]  <- KHÔNG phải cột
print([max(col) for col in zip(*matrix)])    # [7, 8, 9]: max từng cột

[(1, 4, 7), (2, -1, 8), (-1, 6, 9)]
[([1, 2, -1],), ([4, -1, 6],), ([7, 8, 9],)]
[7, 8, 9]


### 3.4. Sao chép ma trận: nông và sâu

`matrix[:]` hay `list(matrix)` chỉ sao chép **danh sách ngoài**; các **hàng bên trong vẫn là cùng đối tượng**, nên sửa `copy[i][j]` làm đổi luôn `matrix[i][j]`. Để có bản sao độc lập, sao chép **từng hàng**: `[row[:] for row in matrix]` (hoặc `copy.deepcopy`).

In [5]:
matrix = [[1, 2], [3, 4]]
shallow = matrix[:]
shallow[0][0] = 99
print(matrix)                                # [[99, 2], [3, 4]]  <- ma trận gốc bị đổi!

matrix = [[1, 2], [3, 4]]
deep = [row[:] for row in matrix]
deep[0][0] = 99
print(matrix, deep)                          # [[1, 2], [3, 4]] [[99, 2], [3, 4]]

[[99, 2], [3, 4]]
[[1, 2], [3, 4]] [[99, 2], [3, 4]]


### 3.5. List comprehension lồng nhau

`[[f(i, j) for j in range(n)] for i in range(m)]` tạo ma trận mới: vòng ngoài theo hàng, vòng trong theo cột. Đọc từ **trong ra ngoài** là "với mỗi hàng `i`, tạo một danh sách các ô `j`".

In [6]:
m, n = 2, 3
print([[i * 10 + j for j in range(n)] for i in range(m)])     # [[0, 1, 2], [10, 11, 12]]

[[0, 1, 2], [10, 11, 12]]


### 3.6. `max` có phần tử `-1`

`max` của một cột chứa cả `-1` và các số không âm là một số **không âm**, vì `-1` nhỏ hơn mọi số không âm. Do đề đảm bảo mỗi cột có ít nhất một số không âm, ta **không cần loại `-1`** khi tính max của cột.

In [7]:
print(max([-1, 6, 9]), max([-1, 0, -1]), max([-1, -1, 4]))      # 9 0 4

9 0 4


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Với `matrix = [[1, 2, -1], [4, -1, 6], [7, 8, 9]]`, làm hai bước:

**Bước 1 — tìm max của mỗi cột.**

| | cột 0 | cột 1 | cột 2 |
|---|---|---|---|
| hàng 0 | 1 | 2 | **-1** |
| hàng 1 | 4 | **-1** | 6 |
| hàng 2 | 7 | 8 | 9 |
| **max cột** | **7** | **8** | **9** |

**Bước 2 — thay các ô `-1`** bằng max cột của nó: ô `(0, 2)` thành `9`, ô `(1, 1)` thành `8`. Kết quả `[[1, 2, 9], [4, 8, 6], [7, 8, 9]]`. Các ô khác **giữ nguyên**.

Điều quan trọng: **max cột chỉ cần tính một lần cho mỗi cột**, không phải tính lại cho mỗi ô `-1`.

In [8]:
matrix = [[1, 2, -1], [4, -1, 6], [7, 8, 9]]
m, n = len(matrix), len(matrix[0])
col_max = [max(matrix[i][j] for i in range(m)) for j in range(n)]
print("max của từng cột:", col_max)
for i in range(m):
    for j in range(n):
        if matrix[i][j] == -1:
            print(f"ô ({i}, {j}) = -1 -> thay bằng max cột {j} = {col_max[j]}")

max của từng cột: [7, 8, 9]
ô (0, 2) = -1 -> thay bằng max cột 2 = 9
ô (1, 1) = -1 -> thay bằng max cột 1 = 8


## 5. Cách 1 — Brute Force / baseline: với mỗi ô `-1`, quét cả cột

**Ý tưởng.** Sao chép ma trận. Với mỗi ô `-1` tại `(i, j)`: duyệt cả cột `j` của **ma trận gốc** để tìm max, rồi gán vào `answer[i][j]`.

In [9]:
def modified_matrix_scan(matrix: List[List[int]]) -> List[List[int]]:
    m, n = len(matrix), len(matrix[0])
    answer = [row[:] for row in matrix]                # sao chép từng hàng (độc lập với matrix)
    for i in range(m):
        for j in range(n):
            if matrix[i][j] == -1:
                best = -1
                for r in range(m):                      # quét cả cột j của ma trận gốc
                    best = max(best, matrix[r][j])
                answer[i][j] = best
    return answer


for matrix, expected in examples:
    print(matrix, "->", modified_matrix_scan(matrix), "(kỳ vọng", expected, ")")

[[1, 2, -1], [4, -1, 6], [7, 8, 9]] -> [[1, 2, 9], [4, 8, 6], [7, 8, 9]] (kỳ vọng [[1, 2, 9], [4, 8, 6], [7, 8, 9]] )
[[3, -1], [5, 2]] -> [[3, 2], [5, 2]] (kỳ vọng [[3, 2], [5, 2]] )
[[-1, 0, 5], [-1, 1, -1], [4, -1, 2]] -> [[4, 0, 5], [4, 1, 5], [4, 1, 2]] (kỳ vọng [[4, 0, 5], [4, 1, 5], [4, 1, 2]] )


**Độ phức tạp (Cách 1).** Trong tệ nhất mọi ô là `-1` (trừ vài ô), mỗi ô quét `m` phần tử: **Time `O(m · n · m)`**; bản sao: **Space `O(m · n)`**.

**Nút thắt.** Cùng một cột bị quét **lặp lại** cho mỗi ô `-1` thuộc cột đó, dù max cột là **cố định**.

## 6. Ý tưởng thuật toán / bước đột phá

Max của một cột **chỉ phụ thuộc cột** chứ không phụ thuộc ô `-1` nào đang cần thay. Vậy:

1. Tính **`col_max[j]` cho mọi cột `j`** đúng **một lần**: `O(m · n)`.
2. Duyệt ma trận một lần nữa: ô nào là `-1` thì lấy `col_max[j]`, ô nào khác thì giữ nguyên: `O(m · n)`.

Tổng `O(m · n)` thay vì `O(m² · n)`. Việc tính max cột có thể dùng chuyển vị `zip(*matrix)` cho gọn.

## 7. Các cách tiếp theo

### Cách 2 — Tính `col_max` một lần bằng vòng lặp, rồi thay

**Giải thích.** Mảng `col_max` độ dài `n` khởi tạo `-1`; duyệt mọi ô để cập nhật `col_max[j] = max(col_max[j], matrix[i][j])`. Rồi sao chép ma trận và thay các ô `-1` bằng `col_max[j]`.

In [10]:
def modified_matrix_colmax(matrix: List[List[int]]) -> List[List[int]]:
    m, n = len(matrix), len(matrix[0])
    col_max = [-1] * n                                 # col_max[j] = max của cột j
    for i in range(m):
        for j in range(n):
            if matrix[i][j] > col_max[j]:
                col_max[j] = matrix[i][j]
    answer = [row[:] for row in matrix]
    for i in range(m):
        for j in range(n):
            if answer[i][j] == -1:
                answer[i][j] = col_max[j]
    return answer


for matrix, expected in examples:
    print(matrix, "->", modified_matrix_colmax(matrix), "(kỳ vọng", expected, ")")

[[1, 2, -1], [4, -1, 6], [7, 8, 9]] -> [[1, 2, 9], [4, 8, 6], [7, 8, 9]] (kỳ vọng [[1, 2, 9], [4, 8, 6], [7, 8, 9]] )
[[3, -1], [5, 2]] -> [[3, 2], [5, 2]] (kỳ vọng [[3, 2], [5, 2]] )
[[-1, 0, 5], [-1, 1, -1], [4, -1, 2]] -> [[4, 0, 5], [4, 1, 5], [4, 1, 2]] (kỳ vọng [[4, 0, 5], [4, 1, 5], [4, 1, 2]] )


**Độ phức tạp Cách 2:** **Time `O(m · n)`**, **Space `O(n)`** cho `col_max` (cộng bản sao đầu ra `O(m · n)`).

### Cách 3 — `zip(*matrix)` lấy cột, list comprehension dựng ma trận mới

**Giải thích.** `col_max = [max(col) for col in zip(*matrix)]` (mục 3.3). Rồi dựng ma trận mới: với mỗi ô `v` ở cột `j`, lấy `col_max[j]` nếu `v == -1`, ngược lại `v`. Một biểu thức, không cần sao chép riêng.

In [11]:
def modified_matrix_zip(matrix: List[List[int]]) -> List[List[int]]:
    col_max = [max(col) for col in zip(*matrix)]       # zip(*matrix): các cột
    return [[col_max[j] if v == -1 else v for j, v in enumerate(row)] for row in matrix]


for matrix, expected in examples:
    print(matrix, "->", modified_matrix_zip(matrix), "(kỳ vọng", expected, ")")

# Quan sát các cột
print(list(zip(*examples[0][0])))

[[1, 2, -1], [4, -1, 6], [7, 8, 9]] -> [[1, 2, 9], [4, 8, 6], [7, 8, 9]] (kỳ vọng [[1, 2, 9], [4, 8, 6], [7, 8, 9]] )
[[3, -1], [5, 2]] -> [[3, 2], [5, 2]] (kỳ vọng [[3, 2], [5, 2]] )
[[-1, 0, 5], [-1, 1, -1], [4, -1, 2]] -> [[4, 0, 5], [4, 1, 5], [4, 1, 2]] (kỳ vọng [[4, 0, 5], [4, 1, 5], [4, 1, 2]] )
[(1, 4, 7), (2, -1, 8), (-1, 6, 9)]


**Độ phức tạp Cách 3:** **Time `O(m · n)`**, **Space `O(m · n)`** cho ma trận kết quả (`col_max` chỉ `O(n)`).

### Cách 4 — Sửa tại chỗ (in-place), không dựng ma trận mới

**Giải thích.** Tính `col_max` như Cách 2/3 rồi **ghi thẳng** vào `matrix` thay vì tạo bản sao; trả về chính `matrix`. Tiết kiệm bộ nhớ (`O(n)` phụ), nhưng **làm hỏng ma trận đầu vào** (đề gốc cho phép trả lại chính mảng đã sửa, nhưng trong chương trình thật điều này có tác dụng phụ). Ta nên biết cả hai cách và chọn đúng theo yêu cầu.

In [12]:
def modified_matrix_inplace(matrix: List[List[int]]) -> List[List[int]]:
    col_max = [max(col) for col in zip(*matrix)]       # tính trước, trên dữ liệu gốc
    for row in matrix:
        for j in range(len(row)):
            if row[j] == -1:
                row[j] = col_max[j]                    # ghi thẳng vào matrix
    return matrix


for matrix, expected in examples:
    data = [row[:] for row in matrix]                  # dùng bản sao để không phá `examples`
    print(matrix, "->", modified_matrix_inplace(data), "(kỳ vọng", expected, ") | data bị sửa:", data != matrix)

[[1, 2, -1], [4, -1, 6], [7, 8, 9]] -> [[1, 2, 9], [4, 8, 6], [7, 8, 9]] (kỳ vọng [[1, 2, 9], [4, 8, 6], [7, 8, 9]] ) | data bị sửa: True
[[3, -1], [5, 2]] -> [[3, 2], [5, 2]] (kỳ vọng [[3, 2], [5, 2]] ) | data bị sửa: True
[[-1, 0, 5], [-1, 1, -1], [4, -1, 2]] -> [[4, 0, 5], [4, 1, 5], [4, 1, 2]] (kỳ vọng [[4, 0, 5], [4, 1, 5], [4, 1, 2]] ) | data bị sửa: True


**Độ phức tạp Cách 4:** **Time `O(m · n)`**, **Space `O(n)`** phụ (chỉ `col_max`).

In [13]:
# Code nộp LeetCode cho Cách 3 (ngắn, tạo ma trận mới đúng như đề)
from typing import List

class Solution:
    def modifiedMatrix(self, matrix: List[List[int]]) -> List[List[int]]:
        col_max = [max(col) for col in zip(*matrix)]
        return [[col_max[j] if v == -1 else v for j, v in enumerate(row)] for row in matrix]

## 8. TRACE từng bước

Ta theo dõi (a) Cách 2 (tính `col_max` rồi thay) trên ví dụ 1 và (b) Cách 3 từng hàng.

In [14]:
def trace_colmax(matrix):
    m, n = len(matrix), len(matrix[0])
    col_max = [-1] * n
    print("matrix =", matrix)
    for i in range(m):
        for j in range(n):
            if matrix[i][j] > col_max[j]:
                col_max[j] = matrix[i][j]
        print(f"sau hàng {i}: col_max = {col_max}")
    answer = [row[:] for row in matrix]
    for i in range(m):
        for j in range(n):
            if answer[i][j] == -1:
                answer[i][j] = col_max[j]
                print(f"ô ({i}, {j}) là -1 -> thay bằng col_max[{j}] = {col_max[j]}")
    print("=> kết quả:", answer)
    return answer


assert trace_colmax([[1, 2, -1], [4, -1, 6], [7, 8, 9]]) == [[1, 2, 9], [4, 8, 6], [7, 8, 9]]

matrix = [[1, 2, -1], [4, -1, 6], [7, 8, 9]]
sau hàng 0: col_max = [1, 2, -1]
sau hàng 1: col_max = [4, 2, 6]
sau hàng 2: col_max = [7, 8, 9]
ô (0, 2) là -1 -> thay bằng col_max[2] = 9
ô (1, 1) là -1 -> thay bằng col_max[1] = 8
=> kết quả: [[1, 2, 9], [4, 8, 6], [7, 8, 9]]


In [15]:
def trace_zip(matrix):
    cols = list(zip(*matrix))
    col_max = [max(c) for c in cols]
    print("các cột    :", cols)
    print("max từng cột:", col_max)
    result = []
    for i, row in enumerate(matrix):
        new_row = [col_max[j] if v == -1 else v for j, v in enumerate(row)]
        print(f"hàng {i}: {row} -> {new_row}")
        result.append(new_row)
    return result


assert trace_zip([[3, -1], [5, 2]]) == [[3, 2], [5, 2]]

các cột    : [(3, 5), (-1, 2)]
max từng cột: [5, 2]
hàng 0: [3, -1] -> [3, 2]
hàng 1: [5, 2] -> [5, 2]


Bảng tóm tắt (Cách 2, ví dụ 1):

| bước | nội dung |
|---|---|
| sau hàng 0 | `col_max = [1, 2, -1]` |
| sau hàng 1 | `col_max = [4, 2, 6]` |
| sau hàng 2 | `col_max = [7, 8, 9]` |
| thay | ô `(0, 2)` bằng `9`; ô `(1, 1)` bằng `8` |

## 9. Code nộp LeetCode

Bản nộp dùng Cách 3 (`zip(*matrix)` lấy max cột, list comprehension dựng ma trận mới). Đúng class `Solution`, đúng method `modifiedMatrix(self, matrix: List[List[int]]) -> List[List[int]]`.

In [16]:
from typing import List

class Solution:
    def modifiedMatrix(self, matrix: List[List[int]]) -> List[List[int]]:
        col_max = [max(col) for col in zip(*matrix)]            # max của từng cột (zip(*matrix) cho các cột)
        return [
            [col_max[j] if v == -1 else v for j, v in enumerate(row)]   # -1 thành max cột, ô khác giữ nguyên
            for row in matrix
        ]


sol = Solution()
for matrix, expected in examples:
    print(matrix, "->", sol.modifiedMatrix(matrix))

[[1, 2, -1], [4, -1, 6], [7, 8, 9]] -> [[1, 2, 9], [4, 8, 6], [7, 8, 9]]
[[3, -1], [5, 2]] -> [[3, 2], [5, 2]]
[[-1, 0, 5], [-1, 1, -1], [4, -1, 2]] -> [[4, 0, 5], [4, 1, 5], [4, 1, 2]]


## 10. Vì sao thuật toán đúng

- **Max cột không bị ảnh hưởng bởi việc thay.** Giá trị thay thế là max của cột trong **ma trận gốc**. Vì ta tính `col_max` **trước** khi thay (hoặc dùng ma trận gốc), việc thay một ô `-1` không đổi max các cột khác, cũng không đổi max cột của chính nó.
- **Max cột luôn không âm.** Đề đảm bảo mỗi cột có ít nhất một ô không âm; mà `-1 <` mọi số không âm nên `max(col) >= 0`. Do đó phép thay không thể tạo ra `-1` mới.
- **Ô khác giữ nguyên.** Chỉ ô có giá trị `-1` được thay; mọi ô còn lại (`0..100`) được sao chép đúng.
- **Độc lập với ma trận gốc.** Dựng ma trận mới bằng comprehension tạo các hàng **mới** (không dùng chung với `matrix`), nên không ảnh hưởng đầu vào.
- **Duyệt đủ ô.** Hai vòng `for row` / `enumerate(row)` đi qua đúng `m · n` ô một lần, cho kết quả cùng kích thước.

## 11. Phân tích Time / Space Complexity

Gọi `m` hàng, `n` cột.

- **Cách 1 (quét cột cho mỗi ô `-1`):** Time `O(m² · n)`, Space `O(m · n)` (bản sao).
- **Cách 2 (`col_max` bằng vòng lặp):** Time `O(m · n)`, Space `O(m · n)` (bản sao) + `O(n)`.
- **Cách 3 (`zip(*matrix)` + comprehension):** Time `O(m · n)`, Space `O(m · n)` (kết quả) + `O(n)`.
- **Cách 4 (sửa tại chỗ):** Time `O(m · n)`, Space `O(n)` phụ.
- Phải đọc mọi ô nên `O(m · n)` là tối ưu. Ở đây `m, n <= 50`: tối đa `2 500` ô.

## 12. Bộ test

In [17]:
sol = Solution()

def inplace_on_copy(matrix):
    return modified_matrix_inplace([row[:] for row in matrix])

impls = {
    "scan": modified_matrix_scan,
    "colmax": modified_matrix_colmax,
    "zip": modified_matrix_zip,
    "inplace(copy)": inplace_on_copy,
    "submit": sol.modifiedMatrix,
}

for matrix, expected in examples:
    for key, fn in impls.items():
        got = fn([row[:] for row in matrix])
        assert got == expected, (key, matrix, got, expected)
print("PASS ví dụ của đề (mọi cách)")

PASS ví dụ của đề (mọi cách)


### 12.1. Các trường hợp biên

In [18]:
def check_all(matrix, expected, label):
    for key, fn in impls.items():
        got = fn([row[:] for row in matrix])
        assert got == expected, (label, key, got, expected)
    print("PASS", label, "->", expected)


check_all([[1, 2], [3, 4]], [[1, 2], [3, 4]], "không có ô -1: giữ nguyên")
check_all([[-1, 5], [2, -1]], [[2, 5], [2, 5]], "hai ô -1 ở hai cột khác nhau")
check_all([[-1, -1], [3, 4]], [[3, 4], [3, 4]], "cả hàng đầu là -1")
check_all([[3, 4], [-1, -1]], [[3, 4], [3, 4]], "cả hàng cuối là -1")
check_all([[0, -1], [-1, 0]], [[0, 0], [0, 0]], "max cột là 0 (giá trị không âm nhỏ nhất)")
check_all([[100, -1], [-1, 100]], [[100, 100], [100, 100]], "giá trị lớn nhất 100")
check_all([[5, -1, 7], [-1, 6, -1]], [[5, 6, 7], [5, 6, 7]], "ma trận 2 x 3 (không vuông)")
check_all([[-1, 2], [3, -1], [-1, 4]], [[3, 2], [3, 4], [3, 4]], "ma trận 3 x 2 (không vuông)")
check_all([[7, -1], [-1, 9], [2, 3]], [[7, 9], [7, 9], [2, 3]], "ô -1 ở hàng giữa")
check_all([[-1] + [5] * 49] + [[1] * 50 for _ in range(49)], [[1] + [5] * 49] + [[1] * 50 for _ in range(49)], "50 x 50: một ô -1 ở góc")

PASS không có ô -1: giữ nguyên -> [[1, 2], [3, 4]]
PASS hai ô -1 ở hai cột khác nhau -> [[2, 5], [2, 5]]
PASS cả hàng đầu là -1 -> [[3, 4], [3, 4]]
PASS cả hàng cuối là -1 -> [[3, 4], [3, 4]]
PASS max cột là 0 (giá trị không âm nhỏ nhất) -> [[0, 0], [0, 0]]
PASS giá trị lớn nhất 100 -> [[100, 100], [100, 100]]
PASS ma trận 2 x 3 (không vuông) -> [[5, 6, 7], [5, 6, 7]]
PASS ma trận 3 x 2 (không vuông) -> [[3, 2], [3, 4], [3, 4]]
PASS ô -1 ở hàng giữa -> [[7, 9], [7, 9], [2, 3]]
PASS 50 x 50: một ô -1 ở góc -> [[1, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1

### 12.2. Tính chất và phản ví dụ

In [19]:
def random_matrix(rng, m=None, n=None, p_neg=0.4):
    m = m or rng.randint(2, 50)
    n = n or rng.randint(2, 50)
    mat = [[-1 if rng.random() < p_neg else rng.randint(0, 100) for _ in range(n)] for _ in range(m)]
    for j in range(n):                                          # đảm bảo mỗi cột có ít nhất một số không âm
        if all(mat[i][j] == -1 for i in range(m)):
            mat[rng.randrange(m)][j] = rng.randint(0, 100)
    return mat


rng = random.Random(11)
# Tính chất 1: không còn -1; kích thước giữ nguyên; ô không phải -1 giữ nguyên; ô -1 bằng max cột
for _ in range(300):
    mat = random_matrix(rng)
    out = sol.modifiedMatrix(mat)
    m, n = len(mat), len(mat[0])
    assert len(out) == m and all(len(r) == n for r in out)
    for j in range(n):
        cmax = max(mat[i][j] for i in range(m))
        for i in range(m):
            assert out[i][j] == (cmax if mat[i][j] == -1 else mat[i][j])
    assert all(v >= 0 for row in out for v in row)
print("PASS không còn -1; ô khác giữ nguyên; ô -1 bằng max cột")

# Tính chất 2: max của từng cột không đổi sau khi thay
for _ in range(300):
    mat = random_matrix(rng)
    out = sol.modifiedMatrix(mat)
    assert [max(c) for c in zip(*mat)] == [max(c) for c in zip(*out)]
print("PASS max từng cột không đổi")

# Tính chất 3: lũy đẳng (áp dụng lại không đổi gì)
for _ in range(300):
    mat = random_matrix(rng)
    out = sol.modifiedMatrix(mat)
    assert sol.modifiedMatrix(out) == out
print("PASS lũy đẳng: thay lần hai không đổi gì")

# Tính chất 4: các cách sao chép/ không sửa đầu vào (trừ Cách 4 trực tiếp)
mat = [[1, 2, -1], [4, -1, 6], [7, 8, 9]]
copy = [row[:] for row in mat]
for key, fn in impls.items():
    fn(mat)
assert mat == copy
print("PASS đầu vào không bị sửa (các cách dùng bản sao)")

# Tính chất 5: ma trận kết quả không dùng chung hàng với đầu vào
out = sol.modifiedMatrix(mat)
assert all(out[i] is not mat[i] for i in range(len(mat)))
print("PASS các hàng của kết quả là đối tượng mới")

# Phản ví dụ 1: lấy max theo hàng thay vì theo cột
def sai_hang(matrix):
    return [[max(row) if v == -1 else v for v in row] for row in matrix]

assert sai_hang([[1, 2, -1], [4, -1, 6], [7, 8, 9]]) != [[1, 2, 9], [4, 8, 6], [7, 8, 9]]
print("PASS phản ví dụ: max theo hàng cho", sai_hang([[1, 2, -1], [4, -1, 6], [7, 8, 9]]))

# Phản ví dụ 2: sao chép nông làm hỏng đầu vào
def sai_nong(matrix):
    answer = matrix[:]
    col_max = [max(c) for c in zip(*matrix)]
    for i in range(len(matrix)):
        for j in range(len(matrix[0])):
            if matrix[i][j] == -1:
                answer[i][j] = col_max[j]
    return answer

data = [[3, -1], [5, 2]]
sai_nong(data)
assert data == [[3, 2], [5, 2]]
print("PASS phản ví dụ: sao chép nông đã sửa luôn ma trận gốc ->", data)

PASS không còn -1; ô khác giữ nguyên; ô -1 bằng max cột


PASS max từng cột không đổi
PASS lũy đẳng: thay lần hai không đổi gì
PASS đầu vào không bị sửa (các cách dùng bản sao)
PASS các hàng của kết quả là đối tượng mới
PASS phản ví dụ: max theo hàng cho [[1, 2, 2], [4, 6, 6], [7, 8, 9]]
PASS phản ví dụ: sao chép nông đã sửa luôn ma trận gốc -> [[3, 2], [5, 2]]


### 12.3. Đối chiếu ngẫu nhiên (có seed)

In [20]:
rng = random.Random(3033)
for trial in range(1500):
    shape = rng.choice([(None, None), (2, 50), (50, 2), (2, 2), (7, 3), (3, 7)])
    mat = random_matrix(rng, shape[0], shape[1], p_neg=rng.choice([0.0, 0.2, 0.5, 0.9]))
    ref = modified_matrix_scan([row[:] for row in mat])
    for key, fn in impls.items():
        assert fn([row[:] for row in mat]) == ref, (key, mat)
print("PASS 1500 ma trận ngẫu nhiên (vuông và không vuông) - mọi cách khớp nhau")

PASS 1500 ma trận ngẫu nhiên (vuông và không vuông) - mọi cách khớp nhau


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Lấy max theo hàng thay vì theo cột

In [21]:
def sai_max_hang(matrix):
    return [[max(row) if v == -1 else v for v in row] for row in matrix]


m1 = [[1, 2, -1], [4, -1, 6], [7, 8, 9]]
print("sai  :", sai_max_hang(m1))
print("đúng:", Solution().modifiedMatrix(m1))

sai  : [[1, 2, 2], [4, 6, 6], [7, 8, 9]]
đúng: [[1, 2, 9], [4, 8, 6], [7, 8, 9]]


**Vì sao sai:** đề cần **max của cột**, không phải của hàng. Ô `(0, 2)` thuộc cột `2` (`[-1, 6, 9]`, max `9`), nhưng max của hàng `0` là `2`. Muốn max cột phải duyệt `matrix[i][j]` qua mọi `i` hoặc dùng `zip(*matrix)`.

### Lỗi 2: Sao chép nông làm hỏng ma trận gốc

In [22]:
def sai_sao_chep_nong(matrix):
    answer = matrix[:]                                  # chỉ sao chép danh sách ngoài
    col_max = [max(c) for c in zip(*matrix)]
    for i in range(len(matrix)):
        for j in range(len(matrix[0])):
            if answer[i][j] == -1:
                answer[i][j] = col_max[j]               # sửa luôn hàng dùng chung với matrix
    return answer


data = [[3, -1], [5, 2]]
res = sai_sao_chep_nong(data)
print("kết quả:", res, "| ma trận gốc sau khi gọi:", data)

kết quả: [[3, 2], [5, 2]] | ma trận gốc sau khi gọi: [[3, 2], [5, 2]]


**Vì sao sai:** `matrix[:]` tạo danh sách ngoài mới nhưng các **hàng** bên trong là **cùng đối tượng** với `matrix`. Sửa `answer[i][j]` là sửa luôn `matrix[i][j]`: ma trận gốc bị hỏng (`-1` biến mất). Đề yêu cầu `answer` là **bản sao**; dùng `[row[:] for row in matrix]`.

### Lỗi 3: Nhầm `m` và `n` khi ma trận không vuông

In [23]:
def sai_m_n(matrix):
    m, n = len(matrix), len(matrix[0])
    col_max = [max(matrix[i][j] for i in range(n)) for j in range(m)]       # hoán đổi n và m
    return col_max


try:
    print(sai_m_n([[5, -1, 7], [-1, 6, -1]]))                # 2 hàng, 3 cột
except IndexError as e:
    print("IndexError:", e)

IndexError: list index out of range


**Vì sao sai:** số **hàng** là `m = len(matrix)`, số **cột** là `n = len(matrix[0])`. Duyệt cột `j` thì `j` chạy đến `n` và `i` (hàng) chạy đến `m`. Đảo hai cận gây `IndexError` (hoặc bỏ sót ô) khi `m != n`; ma trận vuông che giấu lỗi này, hãy luôn thử ma trận **không vuông**.

### Lỗi 4: `zip(matrix)` thay vì `zip(*matrix)`

In [24]:
def sai_zip(matrix):
    return [max(col) for col in zip(matrix)]                # thiếu dấu *: mỗi "cột" thực ra là một hàng


print("sai  :", sai_zip([[1, 2], [3, 4]]))
print("đúng:", [max(col) for col in zip(*[[1, 2], [3, 4]])])

sai  : [[1, 2], [3, 4]]
đúng: [3, 4]


**Vì sao sai:** `zip(matrix)` nhận **một** đối số (cả ma trận) nên ghép từng **hàng** thành tuple một phần tử; `max` của mỗi tuple đó chỉ là chính hàng (so sánh danh sách), không phải max cột. Phải **giải nén** các hàng bằng `*`: `zip(*matrix)`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Thấy "max / min / tổng của **cột**" → tính **một lần cho mọi cột** (`col_max`), không tính lại cho từng ô. Cột lấy bằng `zip(*matrix)`.
- Công thức nhớ: **`[max(col) for col in zip(*matrix)]`** cho max từng cột; `[max(row) for row in matrix]` cho max từng hàng.
- "Tạo bản sao rồi sửa" → **sao chép từng hàng**: `[row[:] for row in matrix]`.
- Luôn kiểm thử ma trận **không vuông** (`2 x 3`, `3 x 2`) để bắt lỗi nhầm `m`, `n`.
- `-1` nhỏ hơn mọi số không âm nên `max` bỏ qua nó khi cột có ít nhất một số không âm; đề đã đảm bảo điều đó.
- Sửa tại chỗ tiết kiệm bộ nhớ nhưng có tác dụng phụ; chỉ dùng khi đề cho phép.

## 15. Biến thể của bài

1. Thay `-1` bằng **max của hàng** (thay vì cột).
2. Thay `-1` bằng **min không âm** của cột.
3. Thay `-1` bằng **trung bình** (làm tròn xuống) của các số không âm trong cột.
4. Chỉ thay `-1` nếu cột đó có **đúng một** số `-1`.
5. Điền mỗi ô `-1` bằng **số không âm gần nhất ở bên trái** trong cùng hàng (nếu không có thì giữ `-1`).

Thử biến thể 1, 2 và 5:

In [25]:
def replace_with_row_max(matrix):
    return [[max(row) if v == -1 else v for v in row] for row in matrix]


def replace_with_col_min_nonneg(matrix):
    col_min = [min(v for v in col if v >= 0) for col in zip(*matrix)]
    return [[col_min[j] if v == -1 else v for j, v in enumerate(row)] for row in matrix]


def fill_from_left(matrix):
    out = []
    for row in matrix:
        last, new_row = -1, []
        for v in row:
            if v == -1:
                new_row.append(last)
            else:
                last = v
                new_row.append(v)
        out.append(new_row)
    return out


mat = [[1, 2, -1], [4, -1, 6], [7, 8, 9]]
print("1.", replace_with_row_max(mat))                 # [[1, 2, 2], [4, 6, 6], [7, 8, 9]]
print("2.", replace_with_col_min_nonneg(mat))          # [[1, 2, 6], [4, 2, 6], [7, 8, 9]]
print("5.", fill_from_left([[3, -1, -1, 5], [-1, 2, -1, 1]]))   # [[3, 3, 3, 5], [-1, 2, 2, 1]]
assert replace_with_row_max(mat) == [[1, 2, 2], [4, 6, 6], [7, 8, 9]]
assert replace_with_col_min_nonneg(mat) == [[1, 2, 6], [4, 2, 6], [7, 8, 9]]
assert fill_from_left([[3, -1, -1, 5], [-1, 2, -1, 1]]) == [[3, 3, 3, 5], [-1, 2, 2, 1]]

1. [[1, 2, 2], [4, 6, 6], [7, 8, 9]]
2. [[1, 2, 6], [4, 2, 6], [7, 8, 9]]
5. [[3, 3, 3, 5], [-1, 2, 2, 1]]


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Quét cả cột cho mỗi ô `-1` | định nghĩa trực tiếp | `O(m²·n)` | `O(m·n)` | hiểu đề, đáp án chuẩn |
| 2. `col_max` bằng vòng lặp | tính max cột một lần | `O(m·n)` | `O(m·n)` + `O(n)` | rõ cơ chế, không dùng `zip` |
| 3. `zip(*matrix)` + comprehension | cột bằng chuyển vị | `O(m·n)` | `O(m·n)` | **cách nộp bài**, ngắn gọn |
| 4. Sửa tại chỗ | ghi thẳng vào `matrix` | `O(m·n)` | `O(n)` | khi cho phép sửa đầu vào, tiết kiệm bộ nhớ |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `list(zip(*[[1, 2], [3, 4]]))`, `[max(c) for c in zip(*[[1, 5], [4, 2]])]`, `max([-1, 3, 0])`, `Solution().modifiedMatrix([[2, -1], [-1, 3]])`, `[row[:] for row in [[1], [2]]]`.

**Bài 2 (tính tay).** Với `matrix = [[-1, 5, 2], [3, -1, -1], [4, 6, 1]]`: viết ra từng cột, tìm `col_max`, rồi tính ma trận kết quả.

**Bài 3 (điền TODO).** Hoàn thành `modified_matrix_todo(matrix)` theo Cách 2.

In [26]:
def modified_matrix_todo(matrix):
    m, n = len(matrix), len(matrix[0])
    col_max = [-1] * n
    # TODO 1: duyệt mọi ô (i, j) và cập nhật col_max[j] = max(col_max[j], matrix[i][j])
    answer = [row[:] for row in matrix]
    # TODO 2: với mỗi ô (i, j) mà answer[i][j] == -1, gán answer[i][j] = col_max[j]
    return answer

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (các TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `modified_matrix_zip2(matrix)` theo **Cách 3** (`zip(*matrix)` + comprehension) mà không nhìn code mẫu. Kiểm tra với ba ví dụ trong `examples`.

**Bài 5 (biến thể / mini challenge).** (a) Viết `replace_with_col_avg(matrix)`: thay `-1` bằng **trung bình làm tròn xuống** của các số không âm trong cột. (b) Viết `replace_if_single(matrix)`: chỉ thay `-1` ở những cột có **đúng một** số `-1`. (c) Viết `fill_nearest(matrix)`: thay mỗi `-1` bằng số không âm **gần nhất theo hàng** (trái hoặc phải; hòa thì lấy bên trái), nếu cả hàng toàn `-1` thì giữ nguyên.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `[(1, 3), (2, 4)]`, `[4, 5]`, `3`, `[[2, 3], [2, 3]]`, `[[1], [2]]`. Bài 3: hai vòng lồng nhau cho TODO 1 và 2.
- **Mức 2 (vừa):** Bài 2: các cột `(-1, 3, 4)`, `(5, -1, 6)`, `(2, -1, 1)` nên `col_max = [4, 6, 2]`. Kết quả: `[[4, 5, 2], [3, 6, 2], [4, 6, 1]]`. Bài 4: `col_max = [max(c) for c in zip(*matrix)]` rồi comprehension.
- **Mức 3 (sâu):** Bài 5(a): với mỗi cột lấy `vals = [v for v in col if v >= 0]` rồi `sum(vals) // len(vals)`. 5(b): đếm `-1` mỗi cột bằng `col.count(-1)`. 5(c): với mỗi hàng, với mỗi ô `-1` tìm chỉ số không âm gần nhất bên trái (`l`) và bên phải (`r`) rồi chọn theo khoảng cách; hòa chọn trái.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [27]:
# Bài 1
assert list(zip(*[[1, 2], [3, 4]])) == [(1, 3), (2, 4)]
assert [max(c) for c in zip(*[[1, 5], [4, 2]])] == [4, 5]
assert max([-1, 3, 0]) == 3
assert Solution().modifiedMatrix([[2, -1], [-1, 3]]) == [[2, 3], [2, 3]]
assert [row[:] for row in [[1], [2]]] == [[1], [2]]

# Bài 2
assert list(zip(*[[-1, 5, 2], [3, -1, -1], [4, 6, 1]])) == [(-1, 3, 4), (5, -1, 6), (2, -1, 1)]
assert Solution().modifiedMatrix([[-1, 5, 2], [3, -1, -1], [4, 6, 1]]) == [[4, 5, 2], [3, 6, 2], [4, 6, 1]]

# Bài 3
def modified_matrix_todo_ans(matrix):
    m, n = len(matrix), len(matrix[0])
    col_max = [-1] * n
    for i in range(m):
        for j in range(n):
            col_max[j] = max(col_max[j], matrix[i][j])
    answer = [row[:] for row in matrix]
    for i in range(m):
        for j in range(n):
            if answer[i][j] == -1:
                answer[i][j] = col_max[j]
    return answer

for matrix, expected in examples:
    assert modified_matrix_todo_ans(matrix) == expected

# Bài 4
def modified_matrix_zip2(matrix):
    col_max = [max(c) for c in zip(*matrix)]
    return [[col_max[j] if v == -1 else v for j, v in enumerate(row)] for row in matrix]

for matrix, expected in examples:
    assert modified_matrix_zip2(matrix) == expected

# Bài 5(a)
def replace_with_col_avg(matrix):
    avgs = []
    for col in zip(*matrix):
        vals = [v for v in col if v >= 0]
        avgs.append(sum(vals) // len(vals))
    return [[avgs[j] if v == -1 else v for j, v in enumerate(row)] for row in matrix]

assert replace_with_col_avg([[1, 2], [-1, 4], [5, -1]]) == [[1, 2], [3, 4], [5, 3]]

# Bài 5(b)
def replace_if_single(matrix):
    col_max = [max(c) for c in zip(*matrix)]
    neg_count = [c.count(-1) for c in zip(*matrix)]
    return [[col_max[j] if v == -1 and neg_count[j] == 1 else v for j, v in enumerate(row)] for row in matrix]

assert replace_if_single([[-1, -1, 7], [3, 2, -1], [4, -1, 5]]) == [[4, -1, 7], [3, 2, 7], [4, -1, 5]]    # cột 1 có hai ô -1: giữ nguyên
assert replace_if_single([[1, 2], [3, 4]]) == [[1, 2], [3, 4]]

# Bài 5(c)
def fill_nearest(matrix):
    out = []
    for row in matrix:
        n = len(row)
        new_row = row[:]
        for j in range(n):
            if row[j] == -1:
                left = next((k for k in range(j - 1, -1, -1) if row[k] >= 0), None)
                right = next((k for k in range(j + 1, n) if row[k] >= 0), None)
                if left is None and right is None:
                    continue
                if right is None or (left is not None and j - left <= right - j):
                    new_row[j] = row[left]
                else:
                    new_row[j] = row[right]
        out.append(new_row)
    return out

assert fill_nearest([[3, -1, -1, 5]]) == [[3, 3, 5, 5]]
assert fill_nearest([[-1, 2, -1]]) == [[2, 2, 2]]
assert fill_nearest([[-1, -1]]) == [[-1, -1]]
print("PASS đáp án phần 19")

PASS đáp án phần 19


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (max từng cột một lần, dựng ma trận mới)

```python
from typing import List

class Solution:
    def modifiedMatrix(self, matrix: List[List[int]]) -> List[List[int]]:
        col_max = [max(col) for col in zip(*matrix)]
        return [[col_max[j] if v == -1 else v for j, v in enumerate(row)] for row in matrix]
```

**Checklist**

- [ ] Lấy max theo **cột** (không phải theo hàng)?
- [ ] Tính `col_max` **một lần** (không quét lại cột cho mỗi ô `-1`)?
- [ ] Tạo bản sao **độc lập** (`[row[:] for row in matrix]` hoặc comprehension), không `matrix[:]`?
- [ ] Dùng `zip(*matrix)` (có dấu `*`) để lấy cột?
- [ ] Phân biệt `m` (số hàng) và `n` (số cột); đã thử ma trận không vuông?
- [ ] Chỉ thay ô `-1`; mọi ô khác giữ nguyên?
- [ ] Skeleton đúng `class Solution` / `def modifiedMatrix(self, matrix: List[List[int]]) -> List[List[int]]`?